### Predict Tourney
Uses the registered model from the training process along with this year's processed data to predict the tourney.

In [0]:
from mlflow.tracking import MlflowClient
import mlflow

#best_rf_model_uri = f"runs:/{best_rf_run_id}/model_RandomForest_basic_deltas"
# Find the best run URI.  This is more complicated due to DB Free's access to disk.
client = MlflowClient()
best_rf_run_id = None
best_exp_id = None
for exp in mlflow.search_experiments():
    run_infos = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="tags.mlflow.runName = 'RandomForest_basic_deltas'",
        max_results=1
    )
    if run_infos:
        best_rf_run_id = run_infos[0].info.run_id
        best_exp_id = exp.experiment_id
        break

if best_rf_run_id:
  best_rf_model_uri = f"runs:/{best_rf_run_id}/model_RandomForest_basic_deltas"
  loaded_model = mlflow.sklearn.load_model(best_rf_model_uri)

In [0]:
# Get the predict data, subset, and predict.
feature_sets = {
    "basic_deltas": ["SeedDelta", "WinDelta", "AdjEMDelta", "SoS_AdjEMDelta"]
}

tourney_df = spark.read.table("marchmadness.gold_data_predict")

X = tourney_df.select(feature_sets["basic_deltas"]).toPandas()

display(X)

y_pred = loaded_model.predict(X)
y_proba = loaded_model.predict_proba(X)[:,1]



In [0]:
# # Merge X, y, y_pred, y_proba into a single dataframe and display.  
results_pdf = tourney_df.toPandas()

results_pdf["y_pred"] = y_pred
results_pdf["y_proba"] = y_proba
  
display(results_pdf)

In [0]:
# Clean up, add team names, and save.
results_df = spark.createDataFrame(results_pdf)
results_df = results_df.withColumnRenamed('y_pred', 'Team1Wins')
results_df = results_df.withColumnRenamed('y_proba', 'ProbTeam1Wins')

# Get the team1 and team 2 IDs from the matchups.
matchups_df = spark.read.table('marchmadness.silver_predict_matchups').select('id', 'Team1Id', 'Team2Id')
display(matchups_df)

results_df = results_df.join(matchups_df, on='id', how='left')
# Get the team name from the KenPom data.
teams_df = spark.read.table("marchmadness.silver_kenpom_predict").select('Team', 'TeamID')
joined_df = results_df.join(teams_df, results_df.Team1Id == teams_df.TeamID, "left")\
                        .drop('Team1Id')\
                        .withColumnRenamed('TeamID', 'Team1ID')\
                        .withColumnRenamed('Team', 'Team1Name')
joined_df = joined_df.join(teams_df, results_df.Team2Id == teams_df.TeamID, "left")\
                        .drop('Team2Id')\
                        .withColumnRenamed('TeamID', 'Team2ID')\
                        .withColumnRenamed('Team', 'Team2Name')

display(joined_df)


In [0]:
joined_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable('marchmadness.gold_predictions')

In [0]:
from pyspark.sql.functions import sum, count

# Find the sum of the wins and probabilities for each team.
summary_df = joined_df.groupBy('Team1ID', 'Team1Name', 'Seed_x')\
                .agg(count('*').alias('GamesPredicted'), sum('Team1Wins').alias('ProbWins'), sum('ProbTeam1Wins').alias('TotalProbability'))

display(summary_df)
    